In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split
import math
import random
from random import uniform

In [8]:
mnist = pd.read_csv('cp_sample.csv', sep=';')

In [9]:
data_train, data_test = train_test_split(mnist, test_size = 0.2, random_state = 4)

In [10]:
from sklearn.ensemble import RandomForestClassifier

In [11]:
r_f = RandomForestClassifier(n_estimators=10, oob_score=True) # this initializes a random forest with 10 trees
r_f.fit(data_train.iloc[:,1:],data_train.label) 
# here we fit this random forest by giving the features of all the
# examples of the training set (columns 1 to last) and their labels

/Users/ardjood/jupyter_env/lib/python3.13/site-packages/sklearn/ensemble/_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


,n_estimators,10
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


In [12]:
pred = r_f.predict(data_train.iloc[:2, 1:])
print("Predictions:", pred)
print("True labels:", data_train.label.iloc[:2].tolist())
print("Correct?", list(pred) == data_train.label.iloc[:2].tolist())

Predictions: [9 3]
True labels: [9, 3]
Correct? True


In [13]:
print("ANSWER: The two first training predictions match the true labels (printed in the cell above).")

ANSWER: The two first training predictions match the true labels (printed in the cell above).


In [14]:
print("Training accuracy:", r_f.score(data_train.iloc[:, 1:], data_train.label))
# ANSWER: with 10 trees the training accuracy is very high (often 1.0). That does not mean the forest generalizes.

Training accuracy: 0.99625


In [15]:
r_f.oob_score_

0.62125

In [16]:
# No validation set: choose the number of trees with the OOB score.
# Use the test set only once, after that choice.
oob_scores = []
models = {}
for n_trees in [10, 25, 50, 75, 100, 150, 200, 300]:
    rf = RandomForestClassifier(n_estimators=n_trees, oob_score=True, random_state=4)
    rf.fit(data_train.iloc[:, 1:], data_train.label)
    oob_scores.append((n_trees, rf.oob_score_))
    models[n_trees] = rf
    print(f"n_estimators={n_trees:3d}  train={rf.score(data_train.iloc[:, 1:], data_train.label):.4f}  oob={rf.oob_score_:.4f}")

best_n = max(oob_scores, key=lambda t: (t[1], -t[0]))[0]
print("Selected n_estimators (best OOB):", best_n)

/Users/ardjood/jupyter_env/lib/python3.13/site-packages/sklearn/ensemble/_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


n_estimators= 10  train=0.9950  oob=0.6338
n_estimators= 25  train=1.0000  oob=0.7425
n_estimators= 50  train=1.0000  oob=0.8063
n_estimators= 75  train=1.0000  oob=0.8313
n_estimators=100  train=1.0000  oob=0.8450
n_estimators=150  train=1.0000  oob=0.8512
n_estimators=200  train=1.0000  oob=0.8638
n_estimators=300  train=1.0000  oob=0.8775
Selected n_estimators (best OOB): 300


In [17]:
best_rf = models[best_n]
test_acc = best_rf.score(data_test.iloc[:, 1:], data_test.label)
print("Test accuracy:", test_acc)
print("Estimated generalization error:", 1 - test_acc)

Test accuracy: 0.89
Estimated generalization error: 0.10999999999999999


In [18]:
print("""ANSWER: Choose the number of trees with the OOB score (no validation set). Test is used only once.
Typical result on this split (random_state=4, 800/200): about 300 trees, test accuracy ≈ 0.89, generalization error ≈ 0.11.
If two forests have the same OOB, keep the smaller one.""")

ANSWER: Choose the number of trees with the OOB score (no validation set). Test is used only once.
Typical result on this split (random_state=4, 800/200): about 300 trees, test accuracy ≈ 0.89, generalization error ≈ 0.11.
If two forests have the same OOB, keep the smaller one.


In [19]:
from skimage.feature import hog

def my_hog(row, ori, cell):
    img = row.iloc[1:].to_numpy().reshape(28, 28)
    return pd.Series(hog(img, orientations=ori, pixels_per_cell=(cell, cell), cells_per_block=(1, 1)))

In [20]:
hog_train = data_train.apply(my_hog, axis=1, args=(8, 14))
hog_test = data_test.apply(my_hog, axis=1, args=(8, 14))
hog_train["label"] = data_train.label.values
hog_test["label"] = data_test.label.values
print(hog_train.shape)  # 32 HOG features + label

(800, 33)


In [21]:
# 14x14 is coarse (only 4 blocks). Also build a finer HOG.
hog_train4 = data_train.apply(my_hog, axis=1, args=(8, 4))
hog_test4 = data_test.apply(my_hog, axis=1, args=(8, 4))
hog_train4["label"] = data_train.label.values
hog_test4["label"] = data_test.label.values
print(hog_train4.shape)

(800, 393)


In [22]:
def select_rf(train, test, n_list=(10, 50, 100, 200)):
    scores = []
    saved = {}
    Xtr, ytr = train.drop(columns="label"), train.label
    Xte, yte = test.drop(columns="label"), test.label
    for n_trees in n_list:
        rf = RandomForestClassifier(n_estimators=n_trees, oob_score=True, random_state=4)
        rf.fit(Xtr, ytr)
        print(f"n_estimators={n_trees:3d}  train={rf.score(Xtr, ytr):.4f}  oob={rf.oob_score_:.4f}")
        scores.append((n_trees, rf.oob_score_))
        saved[n_trees] = rf
    best = max(scores, key=lambda t: (t[1], -t[0]))[0]
    acc = saved[best].score(Xte, yte)
    print("Selected n_estimators:", best)
    print("Test accuracy:", acc)
    print("Estimated generalization error:", 1 - acc)
    return best, acc

print("=== HOG 8 orientations, 14x14 cells ===")
select_rf(hog_train, hog_test)

=== HOG 8 orientations, 14x14 cells ===
n_estimators= 10  train=0.9862  oob=0.5750
n_estimators= 50  train=0.9975  oob=0.6913


/Users/ardjood/jupyter_env/lib/python3.13/site-packages/sklearn/ensemble/_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


n_estimators=100  train=0.9975  oob=0.6987
n_estimators=200  train=0.9975  oob=0.7063
Selected n_estimators: 200
Test accuracy: 0.67
Estimated generalization error: 0.32999999999999996


(200, 0.67)

In [23]:
print("=== HOG 8 orientations, 4x4 cells ===")
select_rf(hog_train4, hog_test4)

=== HOG 8 orientations, 4x4 cells ===
n_estimators= 10  train=0.9975  oob=0.6775
n_estimators= 50  train=1.0000  oob=0.8750


/Users/ardjood/jupyter_env/lib/python3.13/site-packages/sklearn/ensemble/_forest.py:611: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


n_estimators=100  train=1.0000  oob=0.9087
n_estimators=200  train=1.0000  oob=0.9187
Selected n_estimators: 200
Test accuracy: 0.895
Estimated generalization error: 0.10499999999999998


(200, 0.895)

In [24]:
print("""ANSWER:
- Raw pixels + many trees: test ≈ 0.89.
- HOG 14x14 is worse (only 4 blocks, too coarse).
- HOG 4x4 + 200 trees: test ≈ 0.90, the most adapted representation here.
OOB selects the forest; the test set is read only once.""")

ANSWER:
- Raw pixels + many trees: test ≈ 0.89.
- HOG 14x14 is worse (only 4 blocks, too coarse).
- HOG 4x4 + 200 trees: test ≈ 0.90, the most adapted representation here.
OOB selects the forest; the test set is read only once.
